In [ ]:
import os, glob, time
import numpy as np, pandas as pd
from scipy.spatial import cKDTree
from joblib import Parallel, delayed

def find_root():
    for r in ["/kaggle/input/rogii-wellbore-geology-prediction",
              "/kaggle/input/competitions/rogii-wellbore-geology-prediction","data"]+sorted(glob.glob("/kaggle/input/*")):
        if glob.glob(os.path.join(r,"train","*__horizontal_well.csv")): return r
    raise FileNotFoundError("data")
DATA=find_root(); print("DATA:",DATA,flush=True)

wells=[os.path.basename(f).split("__")[0] for f in sorted(glob.glob(f"{DATA}/train/*__horizontal_well.csv"))]
W={}
for w in wells:
    try:
        h=pd.read_csv(f"{DATA}/train/{w}__horizontal_well.csv",usecols=["X","Y","Z","TVT","TVT_input"])
    except Exception: continue
    ps=int(h["TVT_input"].notna().sum())
    if ps<60 or ps>=len(h)-5: continue
    W[w]=dict(X=h["X"].to_numpy(float),Y=h["Y"].to_numpy(float),Z=h["Z"].to_numpy(float),
              TVT=h["TVT"].to_numpy(float),ps=ps)
print("usable",len(W),flush=True)

# structural deviation cloud: S=TVT+Z, minus each well's own PREFIX datum (leak-safe)
ST=10
pxs,pys,pdev,pw=[],[],[],[]
for w,d in W.items():
    S=d["TVT"]+d["Z"]; ps=d["ps"]
    datum=np.median((d["TVT_input"] if False else d["TVT"])[:ps]+d["Z"][:ps])  # prefix S datum
    sl=slice(0,len(S),ST)
    pxs.append(d["X"][sl]); pys.append(d["Y"][sl]); pdev.append(S[sl]-datum); pw.append(np.full(len(S[sl]),w))
    d["datum"]=datum
PX=np.concatenate(pxs); PY=np.concatenate(pys); PDEV=np.concatenate(pdev); PW=np.concatenate(pw)
sc=np.array([PX.std(),PY.std()]); sc=np.where(sc<1e-6,1,sc)
tree=cKDTree(np.column_stack([PX/sc[0],PY/sc[1]]))
print("cloud",len(PX),flush=True)

def predict(w,K=25,FETCH=400):
    d=W[w]; ps=d["ps"]; n=len(d["X"])
    q=np.column_stack([d["X"]/sc[0],d["Y"]/sc[1]])
    dist,idx=tree.query(q,k=min(FETCH,len(PX)),workers=-1)
    dist=np.where(PW[idx]==w,np.inf,dist)
    order=np.argsort(dist,axis=1)[:,:K]
    dk=np.take_along_axis(dist,order,1); ik=np.take_along_axis(idx,order,1)
    valid=np.isfinite(dk); wt=np.where(valid,1.0/(dk+1.0),0.0)
    wsum=wt.sum(1); safe=np.where(wsum>0,wsum,1.0)
    dev_hat=np.where(wsum>0,(PDEV[ik]*wt).sum(1)/safe,np.nan)
    # datum from PREFIX only (leak-safe): median(TVT_input+Z) over known
    kn=slice(0,ps)
    datum=np.nanmedian(d["TVT"][kn]+d["Z"][kn])
    S_hat=datum+dev_hat
    tvt=S_hat-d["Z"]
    # small residual bias on prefix
    bias=np.nanmedian(d["TVT"][kn]-tvt[kn]); bias=bias if np.isfinite(bias) else 0.0
    tvt=tvt+bias
    ev=slice(ps,n)
    pred=np.where(np.isfinite(tvt[ev]),tvt[ev],d["TVT"][ps-1])
    return d["TVT"][ev],pred,np.full(n-ps,d["TVT"][ps-1])

def pooled(pairs,key):
    e=[(y-(p if key=="sp" else c))**2 for y,p,c in pairs]; return float(np.sqrt(np.mean(np.concatenate(e))))

t0=time.time()
for K in [10,25,50]:
    pairs=[predict(w,K=K) for w in W]
    print(f"K={K:3d}  const {pooled(pairs,'const'):.3f}  datum-norm offset-well {pooled(pairs,'sp'):.3f}  ({time.time()-t0:.0f}s)",flush=True)
# test-like subset
try:
    hf=glob.glob("/kaggle/input/rogii-cv-harness/well_table_with_dist.csv")
    if hf:
        wt=pd.read_csv(hf[0]); near=set(wt.loc[wt["dist_test"]<10000,"well"])
        pairs=[(w,predict(w,K=25)) for w in W]
        e_all=[]; e_near=[]; c_near=[]
        for w,(y,p,c) in pairs:
            e_all.append((y-p)**2)
            if w in near: e_near.append((y-p)**2); c_near.append((y-c)**2)
        print("test-like  const %.3f  offset-well %.3f (n=%d)"%(
            np.sqrt(np.mean(np.concatenate(c_near))),np.sqrt(np.mean(np.concatenate(e_near))),len(e_near)),flush=True)
except Exception as ex: print("harness",ex)
print("DONE",flush=True)